# Week 07 - Wednesday Take-Home: Hard NLP Patterns & Aspect-Level Sentiment
**PG Diploma - AI-ML & Agentic AI Engineering - IIT Gandhinagar**  
**Student:** Anirudh Sharma | **Dataset:** ShopSense E-Commerce Reviews (10K)  
**Topics:** Negation, Sarcasm, Code-Mixing, Implicit Sentiment, Comparative, ABSA  
**Due:** Saturday 11:59 PM IST | **Folder:** week07/wednesday/

---


## 0. Imports, Constants & Data

In [ ]:
import re
import numpy as np
import pandas as pd
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# -- Constants --
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

NEGATION_CUES   = ['not','no','never','neither','nor','none','nothing',
                   'without','barely','hardly','scarcely','rarely']
NEGATION_WINDOW = 3   # Words to flip after a negation cue

SARCASM_PUNCTUATION_THRESHOLD = 2   # Exclamation/question marks
SARCASM_POSITIVE_WORDS = ['great','wow','amazing','excellent','perfect',
                           'wonderful','fantastic','brilliant','awesome']
SARCASM_NEGATIVE_INDICATORS = ['broke','broken','dead','failed','useless',
                                'returned','refund','waste','horrible','terrible']

HINDI_SENTIMENT_LEXICON = {
    'accha': 'good', 'bahut': 'very', 'achha': 'good', 'bekar': 'bad',
    'kharab': 'bad', 'sahi': 'correct', 'galat': 'wrong', 'sundar': 'beautiful',
    'seedha': 'straightforward', 'jaldi': 'fast', 'dheere': 'slow',
    'pasand': 'liked', 'napasand': 'disliked', 'zyada': 'very',
    'thoda': 'little', 'ekdum': 'completely', 'bilkul': 'absolutely'
}

IMPLICIT_RETURN_SIGNALS = [
    'returned','sent back','refunded','exchanged','asked for refund',
    'gave back','returning','had to return'
]
IMPLICIT_NEGATIVE_ACTIONS = [
    'unboxed and immediately','within hours','within minutes','within days',
    'stopped working','never turned on','arrived broken'
]

COMPARATIVE_BRANDS = [
    'samsung','apple','oneplus','xiaomi','realme','oppo','vivo','sony',
    'lg','nokia','motorola','lenovo','asus','hp','dell','boat','jbl'
]

print('Setup complete.')

---
## Q1 - Pipeline for 5 Hard NLP Patterns in Indian E-Commerce Reviews

### Q1(a) - Negation: 'not bad at all' = Positive

In [ ]:
def detect_negation(text, negation_cues=NEGATION_CUES, window=NEGATION_WINDOW):
    """
    Tag tokens that are semantically flipped by a negation cue.
    Returns list of (token, is_negated) tuples.
    Strategy: within WINDOW tokens after a negation cue, mark as negated.
    """
    tokens = re.sub(r'[^a-z\s]', ' ', text.lower()).split()
    tagged = []
    negation_countdown = 0
    for token in tokens:
        if token in negation_cues:
            negation_countdown = window
            tagged.append((token, False, 'NEG_CUE'))
        elif negation_countdown > 0:
            tagged.append((token, True, 'NEGATED'))
            negation_countdown -= 1
            # Reset on sentence boundary punctuation
            if token in ['but','however','although','though']:
                negation_countdown = 0
        else:
            tagged.append((token, False, 'NORMAL'))
    return tagged


def negation_aware_features(text, sentiment_lexicon=None):
    """
    Build negation-aware n-gram features.
    Negated words are prefixed with 'NEG_' to create distinct features.
    E.g., 'not bad' -> ['not', 'NEG_bad'] not ['not', 'bad']
    """
    tagged = detect_negation(text)
    features = []
    for token, is_negated, _ in tagged:
        if is_negated:
            features.append(f'NEG_{token}')
        else:
            features.append(token)
    # Bigrams
    bigrams = [f'{a}_{b}' for a, b in zip(features[:-1], features[1:])]
    return features + bigrams


# Test cases
negation_tests = [
    ('not bad at all', 'POSITIVE (double negation)'),
    ('never disappoints always works', 'POSITIVE'),
    ('not good quality broke easily', 'NEGATIVE (negation + negative)'),
    ('hardly any issues great product', 'POSITIVE (downplayed negation)'),
    ('not bad but not great either', 'MIXED'),
]

print('=== Negation Detection Pipeline ===')
for text, expected in negation_tests:
    tagged     = detect_negation(text)
    features   = negation_aware_features(text)
    neg_tokens = [t for t, is_neg, _ in tagged if is_neg]
    print(f'Text    : {text}')
    print(f'Expected: {expected}')
    print(f'Tagged  : {[(t,lbl) for t,_,lbl in tagged]}')
    print(f'Negated tokens: {neg_tokens}')
    print(f'Features (first 8): {features[:8]}')
    print()

print('=== Baseline Failure Mode ===')
print('Naive bag-of-words sees: ["not", "bad", "at", "all"]')
print('  -> "bad" has negative polarity score = -1')
print('  -> Model predicts NEGATIVE (WRONG)')
print()
print('Negation-aware features: ["not", "NEG_bad", "NEG_at", "NEG_all"]')
print('  -> "NEG_bad" is a distinct feature learned as positive in training')
print('  -> Model correctly predicts POSITIVE')

### Q1(b) - Sarcasm: 'Wow great! Broke on day 1'

In [ ]:
def detect_sarcasm_signals(text,
                            positive_words=SARCASM_POSITIVE_WORDS,
                            negative_indicators=SARCASM_NEGATIVE_INDICATORS,
                            punct_threshold=SARCASM_PUNCTUATION_THRESHOLD):
    """
    Rule-based sarcasm detection for e-commerce reviews.
    Signals:
      1. Hyperbolic positive opener + negative continuation
      2. Excessive punctuation (!!!, ???) with negative content
      3. Positive sentiment word immediately followed by negative event
    Returns dict of detected signals and a sarcasm probability [0,1].
    """
    text_lower = text.lower()
    tokens     = re.sub(r'[^a-z\s]', ' ', text_lower).split()

    signals = {}

    # Signal 1: Hyperbolic opener
    first_5    = tokens[:5]
    has_hype   = any(w in positive_words for w in first_5)
    signals['hyperbolic_opener'] = has_hype

    # Signal 2: Positive word co-occurring with negative event
    has_pos    = any(w in positive_words    for w in tokens)
    has_neg    = any(w in negative_indicators for w in tokens)
    signals['pos_neg_cooccurrence'] = has_pos and has_neg

    # Signal 3: Excessive punctuation
    excl_count = text.count('!')
    ques_count = text.count('?')
    signals['excessive_punctuation'] = (excl_count + ques_count) >= punct_threshold

    # Signal 4: Short positive opener before sentence-boundary negative clause
    sentences  = re.split(r'[.!?]', text_lower)
    if len(sentences) >= 2:
        first_pos = any(w in positive_words for w in sentences[0].split())
        later_neg = any(w in negative_indicators for s in sentences[1:]
                        for w in s.split())
        signals['clause_contrast'] = first_pos and later_neg
    else:
        signals['clause_contrast'] = False

    # Score: fraction of signals triggered
    score = sum(signals.values()) / len(signals)
    return signals, round(score, 2)


sarcasm_tests = [
    ('Wow great! Broke on day 1.',           'SARCASTIC'),
    ('Amazing product! Stopped working after 2 hours.',  'SARCASTIC'),
    ('Excellent packaging. Contents were missing.',      'SARCASTIC'),
    ('Really good quality earbuds sound is fantastic',   'GENUINE POSITIVE'),
    ('Terrible product broke immediately terrible.',     'GENUINE NEGATIVE'),
]

print('=== Sarcasm Detection Pipeline ===')
for text, expected in sarcasm_tests:
    signals, score = detect_sarcasm_signals(text)
    prediction = 'SARCASTIC' if score >= 0.5 else 'GENUINE'
    print(f'Text     : {text}')
    print(f'Expected : {expected}')
    print(f'Signals  : {signals}')
    print(f'Score    : {score}  -->  {prediction}')
    print()

print('=== Baseline Failure Mode ===')
print('Naive model on "Wow great! Broke on day 1":')
print('  "wow" = +1, "great" = +1, "broke" = -1')
print('  Sum = +1 => predicts POSITIVE (WRONG)')
print()
print('Key insight: polarity of sarcasm is INVERTED -- the surface sentiment')
print('(positive words) is the opposite of the intended meaning (negative).')
print('Rule-based signals: hyperbolic opener + negative event = sarcasm flag.')
print('Better approach: train a classifier on (text, context) pairs with sarcasm labels.')

### Q1(c) - Code-Mixing: Hindi-English Reviews

In [ ]:
def transliterate_hindi_tokens(tokens, lexicon=HINDI_SENTIMENT_LEXICON):
    """
    Map Hindi tokens (Roman transliteration) to English equivalents
    using a domain-specific sentiment lexicon.
    Returns (translated_tokens, mapping_dict).
    """
    translated = []
    mapping    = {}
    for token in tokens:
        if token in lexicon:
            translated.append(lexicon[token])
            mapping[token] = lexicon[token]
        else:
            translated.append(token)
    return translated, mapping


def preprocess_code_mixed(text, lexicon=HINDI_SENTIMENT_LEXICON):
    """
    Full preprocessing pipeline for Hindi-English code-mixed text.
    Steps:
      1. Lowercase and tokenise
      2. Translate Hindi tokens to English via lexicon
      3. Apply negation detection on merged text
    Returns (processed_text, token_mapping, final_features).
    """
    tokens = re.sub(r'[^a-z\s]', ' ', text.lower()).split()
    translated_tokens, mapping = transliterate_hindi_tokens(tokens, lexicon)
    processed_text = ' '.join(translated_tokens)
    features       = negation_aware_features(processed_text)
    return processed_text, mapping, features


codemix_tests = [
    ('Product bahut accha hai lekin delivery late thi',
     'Product very good is but delivery late was'),
    ('Camera quality ekdum perfect hai price bhi sahi hai',
     'Camera quality completely perfect is price also correct is'),
    ('Yeh product bilkul bekar hai returned kar diya',
     'This product absolutely bad is returned done'),
    ('Battery bahut jaldi khatam ho jati hai kharab product',
     'Battery very fast finished becomes bad product'),
]

print('=== Code-Mixing Preprocessing Pipeline ===')
for text, expected_translation in codemix_tests:
    processed, mapping, features = preprocess_code_mixed(text)
    print(f'Original : {text}')
    print(f'Mapping  : {mapping}')
    print(f'Processed: {processed}')
    print(f'Features : {features[:8]}')
    print()

print('=== Baseline Failure Mode ===')
print('English-only model on: "Product bahut accha hai lekin delivery late thi"')
print('  Recognised tokens: product, delivery, late')
print('  Unknown tokens:    bahut, accha, hai, lekin, thi  (all dropped)')
print('  Effective text:    "product delivery late"')
print('  Prediction:        NEGATIVE (misses the positive sentiment in "bahut accha")')
print()
print('After Hindi transliteration:')
print('  Text: "product very good is but delivery late was"')
print('  Recognised: product, very, good, delivery, late')
print('  Prediction: MIXED (correctly captures positive product + negative delivery)')

### Q1(d) - Implicit Sentiment: 'Returned it within 2 hours'

In [ ]:
def detect_implicit_sentiment(text,
                               return_signals=IMPLICIT_RETURN_SIGNALS,
                               negative_actions=IMPLICIT_NEGATIVE_ACTIONS):
    """
    Detect implicit negative sentiment from behavioural/factual signals.
    Implicit negative: statements that imply dissatisfaction without
    using opinion words.
    Returns (implicit_sentiment, signals_found, confidence).
    """
    text_lower = text.lower()
    found_signals = []

    # Signal 1: Return behaviour
    for signal in return_signals:
        if signal in text_lower:
            found_signals.append(('RETURN_BEHAVIOUR', signal))

    # Signal 2: Rapid failure (time within X)
    time_pattern = r'within\s+(\w+)\s+(hour|minute|day|week)'
    matches = re.findall(time_pattern, text_lower)
    for qty, unit in matches:
        found_signals.append(('RAPID_FAILURE', f'within {qty} {unit}'))

    # Signal 3: Negative action verbs without explicit opinion
    for action in negative_actions:
        if action in text_lower:
            found_signals.append(('NEGATIVE_ACTION', action))

    # Signal 4: Very short ownership duration before issue
    short_pattern = r'(day|week|hour)\s+(1|one|two|2|3|three|four|4|five|5)'
    if re.search(short_pattern, text_lower):
        found_signals.append(('SHORT_OWNERSHIP', 'implied rapid failure'))

    if found_signals:
        confidence = min(1.0, len(found_signals) * 0.35)
        return 'IMPLICIT_NEGATIVE', found_signals, round(confidence, 2)
    return 'NEUTRAL/UNKNOWN', [], 0.0


implicit_tests = [
    ('Returned it within 2 hours.',
     'IMPLICIT_NEGATIVE: returned + within 2 hours'),
    ('Had to send it back after just one day.',
     'IMPLICIT_NEGATIVE: had to return + one day'),
    ('Stopped working on day 1.',
     'IMPLICIT_NEGATIVE: stopped working'),
    ('Arrived and immediately sent back, asked for refund.',
     'IMPLICIT_NEGATIVE: sent back + refund'),
    ('Great product works perfectly after 3 months.',
     'POSITIVE (no implicit signals)'),
]

print('=== Implicit Sentiment Detection Pipeline ===')
for text, expected in implicit_tests:
    sentiment, signals, conf = detect_implicit_sentiment(text)
    print(f'Text     : {text}')
    print(f'Expected : {expected}')
    print(f'Sentiment: {sentiment}  (confidence: {conf})')
    print(f'Signals  : {signals}')
    print()

print('=== Baseline Failure Mode ===')
print('Lexicon/BOW model on "Returned it within 2 hours":')
print('  No opinion words detected (no good/bad/great/terrible)')
print('  Model output: NEUTRAL (WRONG -- this is strongly negative)')
print()
print('The customer is expressing maximum dissatisfaction through BEHAVIOUR,')
print('not through opinion words. Feature engineering must capture:')
print('  - Action verbs: returned, sent back, exchanged')
print('  - Duration patterns: within X hours/days')
print('  - Implicit failure: stopped working, never turned on')

### Q1(e) - Comparative Sentiment: 'Way better than my previous Samsung'

In [ ]:
COMPARATIVE_MARKERS = [
    'better than', 'worse than', 'superior to', 'inferior to',
    'more than', 'less than', 'way better', 'way worse',
    'far better', 'far worse', 'much better', 'much worse',
    'compared to', 'unlike', 'whereas'
]
POSITIVE_COMPARATIVE = ['better', 'superior', 'way better', 'far better', 'much better']
NEGATIVE_COMPARATIVE = ['worse',  'inferior', 'way worse',  'far worse',  'much worse']


def extract_comparative_sentiment(text,
                                   comparative_markers=COMPARATIVE_MARKERS,
                                   brand_list=COMPARATIVE_BRANDS):
    """
    Detect comparative statements and extract:
      - The polarity for the CURRENT product (positive or negative vs reference)
      - The reference entity (competitor brand or previous product)
      - The aspect being compared (if mentioned)
    Returns dict with comparison details.
    """
    text_lower = text.lower()
    results = []

    for marker in comparative_markers:
        if marker in text_lower:
            # Determine direction
            polarity = 'POSITIVE_FOR_CURRENT'
            for neg_marker in NEGATIVE_COMPARATIVE:
                if neg_marker in text_lower:
                    polarity = 'NEGATIVE_FOR_CURRENT'
                    break

            # Find reference entity
            ref_entity = None
            for brand in brand_list:
                if brand in text_lower:
                    ref_entity = brand
                    break
            if not ref_entity:
                if 'previous' in text_lower or 'old' in text_lower or 'last' in text_lower:
                    ref_entity = 'previous_product'

            # Extract aspect (word before comparative marker)
            pattern = r'(\w+\s+)?' + re.escape(marker)
            match   = re.search(pattern, text_lower)
            aspect  = match.group(1).strip() if match and match.group(1) else 'overall'

            results.append({
                'marker':          marker,
                'polarity':        polarity,
                'reference':       ref_entity,
                'aspect':          aspect,
            })

    return results


comparative_tests = [
    'Way better than my previous Samsung',
    'Camera quality is far superior to OnePlus',
    'Battery life is much worse than my old phone',
    'Unlike Apple, this brand actually delivers on time',
    'Sound quality compared to Sony is not great',
]

print('=== Comparative Sentiment Extraction ===')
for text in comparative_tests:
    results = extract_comparative_sentiment(text)
    print(f'Text   : {text}')
    for r in results:
        print(f'  Marker: {r["marker"]}  |  Polarity: {r["polarity"]}')
        print(f'  Reference: {r["reference"]}  |  Aspect: {r["aspect"]}')
    print()

print('=== Baseline Failure Mode ===')
print('Naive model on "Way better than my previous Samsung":')
print('  Tokens: [way, better, than, my, previous, samsung]')
print('  "better" = positive signal')
print('  Model: POSITIVE (partially correct but wrong reason)')
print()
print('Problem cases:')
print('  "Battery worse than my old Nokia" -> baseline may predict NEGATIVE for Nokia')
print('  "Worse than I expected" -> no reference entity; worse=negative but product is...')
print('The polarity is relative to the CURRENT product, not the reference.')
print('A naive model confuses the comparison direction.')

---
## Q2 - Aspect-Level Sentiment Analysis (ABSA)

### Q2(a) & (b) - Why harder? How to improve?

In [ ]:
print('=== Q2(a): Why is Aspect-Level Harder than Review-Level? ===')
print()
print('Review-Level F1=88%: Classify the WHOLE review as positive/negative/neutral.')
print('Aspect-Level F1=71%: For each ASPECT in the review, classify its sentiment.')
print()
print('Three compounding difficulties:')
print()
print('1. GRANULARITY EXPLOSION')
print('   A single review may contain 3-5 aspects with DIFFERENT polarities.')
print('   Example: "Amazing camera, terrible battery, neutral design"')
print('   Review-level: one label. Aspect-level: 3 separate (aspect, sentiment) pairs.')
print('   Error compounds: wrong aspect boundary + wrong sentiment = double mistake.')
print()
print('2. ASPECT SCOPE AMBIGUITY')
print('   Which sentiment phrase belongs to which aspect?')
print('   "The camera is incredible and the battery life is poor"')
print('   -> camera=positive, battery=negative -- relatively clear.')
print('   "The camera and battery both disappointed me."')
print('   -> ONE opinion spans TWO aspects. Model must split the scope.')
print()
print('3. IMPLICIT ASPECT EXPRESSIONS')
print('   "It drained to 0% before I got home" -- the aspect is battery,')
print('   but the word "battery" never appears. The model must infer it.')
print()
print('4. ANNOTATION DISAGREEMENT')
print('   Human annotators disagree on aspect boundaries more than review labels,')
print('   producing noisier training data that caps F1 around 75-80% for baselines.')

In [ ]:
print('=== Q2(b): How to Improve from 71% to 80%+ ===')
print()
print('Improvement Strategy 1: SPAN-BASED ASPECT EXTRACTION')
print('  Use a sequence labelling model (BERT + CRF or BERT + linear head) to')
print('  identify aspect spans (B-ASP, I-ASP, O tags) before classifying sentiment.')
print('  This separates the two sub-problems: find aspect WHERE, classify polarity HOW.')
print()
print('Improvement Strategy 2: DOMAIN-ADAPTIVE PRE-TRAINING')
print('  Fine-tune BERT on ShopSense unlabelled reviews first (masked LM on domain text)')
print('  before the ABSA fine-tuning. E-commerce vocabulary ("battery drain", "dead pixels")')
print('  is not well-represented in generic BERT pre-training corpora.')
print()
print('Improvement Strategy 3: ASPECT-SPECIFIC ATTENTION')
print('  Given a review + aspect query (e.g., "battery"), use cross-attention')
print('  to let the model focus on the sentiment-bearing tokens for THAT aspect.')
print('  Architecture: AEN-BERT or BERT-PT with aspect-attended encoding.')
print()
print('Improvement Strategy 4: DATA AUGMENTATION FOR RARE ASPECTS')
print('  Aspects like "customer support" or "delivery" have fewer training examples')
print('  than "camera" or "battery". Back-translation or GPT-based paraphrase')
print('  augmentation for rare aspect categories can improve minority-class F1.')
print()
print('Expected gain breakdown:')
rows = [
    ('Span-based extraction (vs keyword matching)',    '+3-4%'),
    ('Domain-adaptive pre-training',                  '+2-3%'),
    ('Aspect-specific attention (AEN-BERT)',           '+3-4%'),
    ('Data augmentation for rare aspects',             '+1-2%'),
    ('Ensemble of rule-based + neural',               '+1%'),
]
print(pd.DataFrame(rows, columns=['Strategy','Expected F1 Gain']).to_string(index=False))
print()
print('Combined: 71 + 10-14 = 81-85% estimated F1 -- exceeds the 80% target.')

### Q2(c) - Aspect-Sentiment Pair Extraction

In [ ]:
# Aspect lexicon: maps aspect keywords to canonical aspect names
ASPECT_LEXICON = {
    'camera': 'CAMERA',    'photo': 'CAMERA',   'photos': 'CAMERA',
    'picture': 'CAMERA',   'image': 'CAMERA',   'shoot': 'CAMERA',
    'battery': 'BATTERY',  'charging': 'BATTERY', 'drain': 'BATTERY',
    'drains': 'BATTERY',   'power': 'BATTERY',  'charge': 'BATTERY',
    'support': 'CUSTOMER_SUPPORT', 'service': 'CUSTOMER_SUPPORT',
    'helpdesk': 'CUSTOMER_SUPPORT', 'response': 'CUSTOMER_SUPPORT',
    'delivery': 'DELIVERY', 'shipping': 'DELIVERY', 'arrived': 'DELIVERY',
    'display': 'DISPLAY',  'screen': 'DISPLAY', 'resolution': 'DISPLAY',
    'sound': 'AUDIO',      'audio': 'AUDIO',    'speaker': 'AUDIO',
    'build': 'BUILD_QUALITY', 'quality': 'BUILD_QUALITY',
    'material': 'BUILD_QUALITY', 'durability': 'BUILD_QUALITY',
}

SENTIMENT_LEXICON = {
    'amazing': 3,    'excellent': 3,  'incredible': 3,  'outstanding': 3,
    'great': 2,      'good': 2,       'wonderful': 2,   'fantastic': 2,
    'decent': 1,     'okay': 1,       'average': 0,     'fine': 1,
    'poor': -2,      'bad': -2,       'terrible': -3,   'atrocious': -3,
    'horrible': -3,  'awful': -3,     'unhelpful': -2,  'useless': -3,
    'slow': -1,      'broken': -3,    'disappointing': -2,
}


def extract_aspect_sentiment_pairs(text,
                                    aspect_lexicon=ASPECT_LEXICON,
                                    sentiment_lexicon=SENTIMENT_LEXICON,
                                    window=5):
    """
    Extract (aspect, sentiment_label, sentiment_score, evidence_phrase) tuples.
    Strategy:
      1. Identify aspect-bearing tokens using aspect_lexicon
      2. For each aspect, find the nearest sentiment word within WINDOW tokens
      3. Apply negation detection to flip sentiment if negated
      4. Map score to label: positive > 0, negative < 0, neutral = 0
    """
    tokens  = re.sub(r'[^a-z\s]', ' ', text.lower()).split()
    n       = len(tokens)
    results = []
    used_aspects = set()

    for i, token in enumerate(tokens):
        if token not in aspect_lexicon:
            continue
        aspect = aspect_lexicon[token]
        if aspect in used_aspects:
            continue
        used_aspects.add(aspect)

        # Search window around the aspect token
        lo = max(0, i - window)
        hi = min(n, i + window + 1)
        window_tokens = tokens[lo:hi]

        # Check for negation in window
        negated = any(t in NEGATION_CUES for t in window_tokens[:window_tokens.index(token)])

        # Find best sentiment word in window
        best_score  = 0
        best_word   = None
        for wt in window_tokens:
            if wt in sentiment_lexicon:
                if abs(sentiment_lexicon[wt]) > abs(best_score):
                    best_score = sentiment_lexicon[wt]
                    best_word  = wt

        if negated and best_score != 0:
            best_score = -best_score
            best_word  = f'NEG_{best_word}'

        label = 'POSITIVE' if best_score > 0 else ('NEGATIVE' if best_score < 0 else 'NEUTRAL')

        # Evidence phrase: aspect token +/- 3 words
        ev_lo = max(0, i-3)
        ev_hi = min(n, i+4)
        evidence = ' '.join(tokens[ev_lo:ev_hi])

        results.append({
            'aspect':    aspect,
            'keyword':   token,
            'sentiment': label,
            'score':     best_score,
            'opinion_word': best_word,
            'evidence':  evidence,
        })

    return results


# The key review from the assignment
review = 'Amazing camera quality but the battery is atrocious and customer support was unhelpful.'

print(f'Review: {review}')
print()
pairs = extract_aspect_sentiment_pairs(review)

print(f'Aspect-Sentiment Pairs Extracted: {len(pairs)}')
print()
for pair in pairs:
    print(f'  ASPECT   : {pair["aspect"]}')
    print(f'  Keyword  : {pair["keyword"]}')
    print(f'  Sentiment: {pair["sentiment"]}  (score: {pair["score"]})')
    print(f'  Opinion  : {pair["opinion_word"]}')
    print(f'  Evidence : "{pair["evidence"]}"')
    print()

print('=== SAME REVIEW = SIMULTANEOUSLY POSITIVE AND NEGATIVE ===')
positive_aspects = [p for p in pairs if p['sentiment'] == 'POSITIVE']
negative_aspects = [p for p in pairs if p['sentiment'] == 'NEGATIVE']
print(f'Positive aspects ({len(positive_aspects)}): {[p["aspect"] for p in positive_aspects]}')
print(f'Negative aspects ({len(negative_aspects)}): {[p["aspect"] for p in negative_aspects]}')
print()
print('This is why review-level sentiment is insufficient for product teams:')
print('Review-level model would predict: MIXED or NEGATIVE (net sentiment).')
print('Aspect-level reveals: camera team should be praised; battery + support teams need action.')
print('One review drives THREE different product team decisions.')

---
## Final Validation Summary

In [ ]:
print('=== Week 07 Wednesday - Validation Checkpoint ===')
print('Q1(a): Negation pipeline       - detect_negation + negation_aware_features')
print(f'       Test: not bad at all   - negated tokens: {[t for t,n,_ in detect_negation("not bad at all") if n]}')
print('Q1(b): Sarcasm detection       - detect_sarcasm_signals with 4 signals')
_, score = detect_sarcasm_signals('Wow great! Broke on day 1.')
print(f'       Wow great! Broke...    - sarcasm score: {score}')
print('Q1(c): Code-mix pipeline       - transliterate_hindi_tokens + preprocess_code_mixed')
proc, mapping, _ = preprocess_code_mixed('Product bahut accha hai lekin delivery late thi')
print(f'       bahut accha translated: {mapping}')
print('Q1(d): Implicit sentiment      - detect_implicit_sentiment with 4 signal types')
sent, sigs, conf = detect_implicit_sentiment('Returned it within 2 hours')
print(f'       Returned within 2hrs   - {sent} (conf={conf})')
print('Q1(e): Comparative extraction  - extract_comparative_sentiment')
res = extract_comparative_sentiment('Way better than my previous Samsung')
print(f'       Way better than Samsung: {res}')
print('Q2(a): Review vs aspect-level  - 3 difficulties explained')
print('Q2(b): Improvement strategies  - 5 strategies, +10-14% F1 estimated')
review_pairs = extract_aspect_sentiment_pairs(
    'Amazing camera quality but the battery is atrocious and customer support was unhelpful.')
print(f'Q2(c): Aspect pairs extracted  - {len(review_pairs)} pairs')
for p in review_pairs:
    print(f'       {p["aspect"]:20s} --> {p["sentiment"]}')
print('\nAll tasks complete. Push week07/wednesday/ to GitHub.')